# Cross-Lingual GraphRAG Pipeline on Kaggle
Ensure you have set the Accelerator to **GPU T4 x2** before running this notebook.
This notebook runs the complete pipeline with **Neo4j Graph Integration** and **Persian Translation**.

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import sys
# Ensure we are in the root working directory and remove any existing repo
os.chdir('/kaggle/working')
!rm -rf /kaggle/working/MedRAG

# Clone the MedRAG repository from GitHub
!git clone https://github.com/TeleEng/MedRAG.git

# Change working directory into the repo
os.chdir('MedRAG')

print("=== Repository Version Info ===")
!git log -1 --format="Commit: %h | Date: %cd"
print("===============================")

# Install dependencies
!pip install -q -r requirements.txt

# Add repo root to Python path so 'from src...' imports work
if '.' not in sys.path:
    sys.path.insert(0, '.')


Cloning into 'MedRAG'...
remote: Enumerating objects: 230, done.
remote: Counting objects: 100% (8/8), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 230 (delta 2), reused 6 (delta 2), pack-reused 222 (from 1)
Receiving objects: 100% (230/230), 439.02 KiB | 3.35 MiB/s, done.
Resolving deltas: 100% (136/136), done.
=== Repository Version Info ===
Commit: 7bccf97 | Date: Sun Aug 30 07:04:36 2026 -0500
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 75.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 992.6/992.6 kB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.5/331.5 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 100.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 92.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━

### Step 0: Neo4j Secrets Configuration
Load Neo4j credentials from Kaggle Secrets so they are available to the pipeline.

In [2]:
from kaggle_secrets import UserSecretsClient
import os

user_secrets = UserSecretsClient()
try:
    os.environ["NEO4J_URI"] = user_secrets.get_secret("NEO4J_URI")
    os.environ["NEO4J_USERNAME"] = user_secrets.get_secret("NEO4J_USERNAME")
    os.environ["NEO4J_PASSWORD"] = user_secrets.get_secret("NEO4J_PASSWORD")
    print("Neo4j Secrets Loaded Successfully!")
except Exception as e:
    print("Warning: Please configure NEO4J_URI, NEO4J_USERNAME, and NEO4J_PASSWORD in Kaggle Secrets.")

Neo4j Secrets Loaded Successfully!


### Step 1: Data Preparation
Download the `medalpaca` dataset and process it into chunks.

In [3]:
from src.data_prep import load_and_prepare_data
load_and_prepare_data()

Loading dataset medalpaca/medical_meadow_wikidoc...


README.md: 0.00B [00:00, ?B/s]

medical_meadow_wikidoc.json:   0%|          | 0.00/10.6M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Processing documents and creating training pairs...
Saved 9373 documents to /kaggle/working/MedRAG/data/processed/documents.json
Saved 9373 training samples to /kaggle/working/MedRAG/data/processed/train_data.json


### Step 2: Indexing (FAISS, BM25, and Neo4j Graph)
Build the Dense/Sparse indexes and push entities to Neo4j AuraDB.

In [4]:
from src.indexer import build_indexes
from src.graph_indexer import GraphIndexer

# 1. Local Indexes
build_indexes()

# 2. Graph Database Indexing
g_indexer = GraphIndexer()
g_indexer.build_graph()
g_indexer.close()

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


Building Dense Index for 9373 documents...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/293 [00:00<?, ?it/s]

Dense Index saved to /kaggle/working/MedRAG/data/processed/faiss_index.bin
Building BM25 Sparse Index...
BM25 Index saved to /kaggle/working/MedRAG/data/processed/bm25_index.pkl
Initializing GraphIndexer in 'auto' mode...
Attempting to connect to Neo4j at neo4j+s://2fa211c8.databases.neo4j.io...
Neo4j connection failed: Failed to DNS resolve address 2fa211c8.databases.neo4j.io:7687: [Errno -2] Name or service not known. Falling back to KùzuDB.
Connecting to local KùzuDB at /kaggle/working/MedRAG/data/processed/kuzu_db...
Creating KùzuDB Schema...
Extracting entities using SpaCy and building Graph (kuzu)...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 47.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel

### Step 3: QLoRA Fine-Tuning
Instruction-tune the base model.

In [5]:
from src.trainer import train_model
train_model()

Loading tokenizer and configuring formatting...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Dropping fully masked examples from train dataset:   0%|          | 0/9373 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Starting QLoRA Fine-Tuning...


Step,Training Loss
10,2.230532
20,2.129481
30,1.966081
40,2.112980
50,2.023689


Saving trained adapter...
Training complete! Run generator.py for end-to-end RAG.


### Step 4: End-to-End Cross-Lingual Generation
Run the full pipeline. The system will translate Persian -> English, query Neo4j+FAISS, generate in English, and translate back to Persian.

In [6]:
from src.generator import MedRAGPipeline

pipeline = MedRAGPipeline()

queries = [
    ("Persian", "علائم بیماری دیابت چیست؟"),
    ("Spanish", "¿Cuáles son los síntomas de la diabetes?"),
    ("French", "Quels sont les symptômes du diabète ?"),
    ("Arabic", "ما هي أعراض مرض السكري؟"),
    ("Chinese", "糖尿病的症状是什么？")
]

for lang, query in queries:
    print(f"\n==========================")
    print(f"TESTING {lang.upper()}")
    print(f"==========================")
    print(f"Query: {query}\n")
    
    res_translated, res_eng, docs = pipeline.answer_query(query)
    
    print(f"\n--- {lang.upper()} TRANSLATED ANSWER ---")
    print(res_translated)
    print(f"\n--- ORIGINAL ENGLISH ANSWER ---")
    print(res_eng)
    print("\n" + "*"*50 + "\n")

Loading documents and indexes...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

...


--- ARABIC TRANSLATED ANSWER ---
وتشمل الأعراض الكلاسيكية للسكري: فقدان الوزن، والكثير من التهاب الكحول، والكثرة من التهاب الكحول، والكثرة من الأعراض الشائعة تشمل تغيرات في الرؤية، والترتيب أو الحساسية في الأطراف، والإرهاق وتغيرات في الجلد.

--- ORIGINAL ENGLISH ANSWER ---
The classic symptoms of diabetes include: weight loss, polyphagia, polydipsia and polyuria. Less common symptoms include vision changes, tingling or numbness in extremities, fatigue and skin changes.

**************************************************


TESTING CHINESE
Query: 糖尿病的症状是什么？


[0] Detecting Language and Translating to English...
Detected Language: zho_Hans

[1] Generating Hypothetical Document (HyDE)...

[2] Executing Hybrid Retrieval with Expanded Query...
Executing Graph Retrieval...


/usr/local/lib/python3.12/dist-packages/spacy/util.py:1800: UserWarning: [W111] Jupyter notebook detected: if using `prefer_gpu()` or `require_gpu()`, include it in the same cell right before `spacy.load()` to ensure that the model is loaded on the correct device. More information: http://spacy.io/usage/v3#jupyter-notebook-gpu
  warnings.warn(Warnings.W111)


Executing FAISS and BM25 Hybrid Search...


Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[3] Translating English Response back to zho_Hans...

--- CHINESE TRANSLATED ANSWER ---
糖尿病的经典症状包括:减肥,多,多和多.不太常见的症状包括视力变化,刺或腿的觉,疲劳和皮肤变化.

--- ORIGINAL ENGLISH ANSWER ---
The classic symptoms of diabetes include: weight loss, polyphagia, polydipsia and polyuria. Less common symptoms include vision changes, tingling or numbness in extremities, fatigue and skin changes.

**************************************************



In [7]:
# --- BILINGUAL CONVERSATIONAL MEMORY TEST ---
# Testing if the pipeline remembers context across language switches
print("\n====================================")
print("TESTING BILINGUAL CONVERSATIONAL MEMORY")
print("====================================")
memory_queries = [
    "what are the diabetes symptoms?",
    "این بیماری چند نوع دارد؟",  # Persian: How many types does this disease have?
    "نوع سوم آن را بیشتر توضیح بده؟",  # Persian: Explain its third type more?
    "Is it lethal?"
]

for query in memory_queries:
    print(f"\nUser: {query}\n")
    res_translated, res_eng, docs = pipeline.answer_query(query)
    print(f"MedRAG: {res_translated}")
    print("-"*50)


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



TESTING BILINGUAL CONVERSATIONAL MEMORY

User: what are the diabetes symptoms?


[0] Detecting Language and Translating to English...
Detected Language: eng_Latn

[1] Generating Hypothetical Document (HyDE)...

[2] Executing Hybrid Retrieval with Expanded Query...
Executing Graph Retrieval...


/usr/local/lib/python3.12/dist-packages/spacy/util.py:1800: UserWarning: [W111] Jupyter notebook detected: if using `prefer_gpu()` or `require_gpu()`, include it in the same cell right before `spacy.load()` to ensure that the model is loaded on the correct device. More information: http://spacy.io/usage/v3#jupyter-notebook-gpu
  warnings.warn(Warnings.W111)


Executing FAISS and BM25 Hybrid Search...


Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...


[3] Translating English Response back to pes_Arab...


Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


MedRAG: نوع سوم سرطان سرطان است که در پکنراس ایجاد می شود.
--------------------------------------------------

User: Is it lethal?


[0] Detecting Language and Translating to English...
Detected Language: eng_Latn

[1] Generating Hypothetical Document (HyDE)...

[2] Executing Hybrid Retrieval with Expanded Query...
Executing Graph Retrieval...


/usr/local/lib/python3.12/dist-packages/spacy/util.py:1800: UserWarning: [W111] Jupyter notebook detected: if using `prefer_gpu()` or `require_gpu()`, include it in the same cell right before `spacy.load()` to ensure that the model is loaded on the correct device. More information: http://spacy.io/usage/v3#jupyter-notebook-gpu
  warnings.warn(Warnings.W111)
Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Executing FAISS and BM25 Hybrid Search...

[3] Translating English Response back to eng_Latn...
MedRAG: Yes
--------------------------------------------------


In [8]:
# --- RAGAS EVALUATION ---
# RAGAS requires an LLM to evaluate the generated answers for hallucinations.
# We will use our local Qwen model to evaluate its own answers.
try:
    from ragas import evaluate
    from ragas.metrics import faithfulness
    from datasets import Dataset
    from langchain_huggingface import HuggingFaceEmbeddings
    import pandas as pd
    
    print("Preparing RAGAS Dataset...")
    test_q = ["What are the symptoms of diabetes?"]
    _, answer, docs = pipeline.answer_query(test_q[0])
    contexts = [[doc.page_content for doc in docs]]
    
    dataset = Dataset.from_dict({
        "question": test_q,
        "answer": [answer],
        "contexts": contexts
    })
    
    print("Initializing Evaluator Embeddings...")
    eval_embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    
    print("Running RAGAS Evaluation (This may take a minute)...")
    result = evaluate(
        dataset, 
        metrics=[faithfulness], 
        llm=pipeline.llm, 
        embeddings=eval_embeddings
    )
    
    print("\n==========================")
    print("RAGAS EVALUATION RESULTS")
    print("==========================")
    print(result)
    
except Exception as e:
    print(f"RAGAS Evaluation encountered an error: {e}")


RAGAS Evaluation encountered an error: No module named 'langchain_community.chat_models.vertexai'
